In [2]:
import seisbench.models as sbm

# List all pretrained weights for PhaseNet
print("PhaseNet pretrained weights:")
for w in sbm.PhaseNet.list_pretrained(details = True):
    print(f"  {w}")

# Same for EQTransformer (good baseline comparison)
print("\nEQTransformer pretrained weights:")
for w in sbm.EQTransformer.list_pretrained(details = True):
    print(f"  {w}")

PhaseNet pretrained weights:
  diting
  ethz
  geofon
  instance
  iquique
  jma
  jma_wc
  lendb
  neic
  obs
  original
  phasenet_sn
  pisdl
  scedc
  stead
  volpick

EQTransformer pretrained weights:
  ethz
  geofon
  instance
  iquique
  lendb
  neic
  obs
  original
  original_nonconservative
  pnw
  scedc
  stead
  volpick


In [3]:
import seisbench.models as sbm
sbm.PhaseNet.list_pretrained(details=True)

{'diting': 'Model trained on the DiTing dataset, a large-scale Chinese seismic benchmark dataset.\nFor models fine-tuned on individual regions in China, please see: https://github.com/JUNZHU-SEIS/USTC-Pickers\nIf you use this model, please reference: Zhu J, Li ZF and Fang LH (2023). USTC-Pickers: a Unified Set of seismic phase pickers Transfer learned for China. Earthq Sci 36(2): 95–112, doi: 10.1016/j.eqs.2023.03.001',
 'ethz': 'Model trained on ETHZ for 100 epochs with a learning rate of 0.01.\nThreshold selected for optimal F1 score on in-domain evaluation. Depending on the target region, the thresholds might need to be adjusted.\nWhen using this model, please reference the SeisBench publications listed at https://github.com/seisbench/seisbench\n\nJannes Münchmeyer, Jack Woollam (munchmej@gfz-potsdam.de, jack.woollam@kit.edu)',
 'geofon': 'Model trained on GEOFON for 100 epochs with a learning rate of 0.001.\nThreshold selected for optimal F1 score on in-domain evaluation. Depending

## see all available datasets. 

In [4]:
import seisbench.data as sbd

# List everything available
print(dir(sbd))

['AQ2009Counts', 'AQ2009GM', 'BenchmarkDataset', 'Bucketer', 'CEED', 'CREW', 'CWA', 'CWANoise', 'ChunkedDummyDataset', 'DummyDataset', 'ETHZ', 'GEOFON', 'GeometricBucketer', 'ISC_EHB_DepthPhases', 'InstanceCounts', 'InstanceCountsCombined', 'InstanceGM', 'InstanceNoise', 'Iquique', 'LFEStacksCascadiaBostock2015', 'LFEStacksMexicoFrank2014', 'LFEStacksSanAndreasShelly2017', 'LenDB', 'MLAAPDE', 'Meier2019JGR', 'MultiWaveformDataset', 'NEIC', 'OBS', 'OBST2024', 'PNW', 'PNWAccelerometers', 'PNWExotic', 'PNWNoise', 'Ross2018GPD', 'Ross2018JGRFM', 'Ross2018JGRPick', 'SCEDC', 'STEAD', 'TXED', 'VCSEIS', 'WaveformDataWriter', 'WaveformDataset', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'aq2009', 'base', 'ceed', 'crew', 'cwa', 'dummy', 'ethz', 'geofon', 'instance', 'iquique', 'isc_ehb', 'lendb', 'lfe_stacks', 'neic', 'obs', 'obst2024', 'pnw', 'scedc', 'stead', 'txed', 'vcseis']


In [5]:
import seisbench.data as sbd
import pandas as pd

candidates = [
    # Tier A benchmark candidates
    'STEAD', 'NEIC', 'GEOFON', 'PNW',
    # OBS
    'OBS', 'OBST2024',
    # Volcanic
    'VCSEIS',
    # Teleseismic depth phases
    'ISC_EHB_DepthPhases',
    # Instance variants
    'InstanceCounts', 'InstanceCountsCombined',
    # Regional (for baseline comparison)
    'SCEDC', 'ETHZ', 'Iquique', 'LenDB', 'TXED',
    # Unknowns — just checking
    'CEED', 'CREW', 'CWA',
]

results = []
for name in candidates:
    try:
        cls = getattr(sbd, name)
        ds = cls(download_kwargs={"metadata_only": True})
        cols = list(ds.metadata.columns)
        has_p = any('p_arrival' in c.lower() or 'p_pick' in c.lower() for c in cols)
        has_s = any('s_arrival' in c.lower() or 's_pick' in c.lower() for c in cols)
        results.append({
            'dataset': name,
            'n_traces': len(ds),
            'has_p_pick': has_p,
            'has_s_pick': has_s,
            'status': 'OK',
            'columns': cols
        })
        print(f"OK  {name:35s}  N={len(ds):>8,}  P={has_p}  S={has_s}")
    except Exception as e:
        results.append({'dataset': name, 'status': str(e)})
        print(f"ERR {name:35s}  {str(e)[:60]}")

# Save so you don't rerun this
df = pd.DataFrame(results)
df.to_csv('seisbench_dataset_inventory.csv', index=False)
print("\nSaved to seisbench_dataset_inventory.csv")

ERR STEAD                                Found partial instance. This suggests that either the downlo


2026-05-05 16:32:02,296 | seisbench | WARNING | Dataset NEIC not in cache.
2026-05-05 16:32:02,298 | seisbench | WARNING | Trying to download preprocessed version from SeisBench repository.
Traceback (most recent call last):
  File "/home/ak287/miniconda3/envs/surface/lib/python3.9/site-packages/tqdm/std.py", line 1144, in __del__
    def __del__(self):
KeyboardInterrupt: 

KeyboardInterrupt



In [6]:
import os
from pathlib import Path

# SeisBench default cache location
cache = Path.home() / ".seisbench" / "datasets"

if cache.exists():
    for d in sorted(cache.iterdir()):
        if d.is_dir():
            files = list(d.iterdir())
            has_meta = any('metadata' in f.name for f in files)
            size_gb = sum(f.stat().st_size for f in files if f.is_file()) / 1e9
            print(f"{d.name:30s}  {size_gb:.1f} GB  metadata={has_meta}")
else:
    print(f"Cache not found at {cache}")

ceed                            0.5 GB  metadata=True
crew                            0.0 GB  metadata=False
dummydataset                    0.0 GB  metadata=True
ethz                            0.0 GB  metadata=True
geofon                          0.1 GB  metadata=True
instancecounts                  1.2 GB  metadata=True
iquique                         0.0 GB  metadata=True
lendb                           16.4 GB  metadata=True
meier2019jgr                    0.0 GB  metadata=False
mlaapde                         0.0 GB  metadata=False
neic                            0.6 GB  metadata=True
obst2024                        0.0 GB  metadata=True
pnw                             0.5 GB  metadata=True
pnwaccelerometers               0.0 GB  metadata=True
ross2018gpd                     0.0 GB  metadata=False
scedc                           2.1 GB  metadata=True
stead                           1.0 GB  metadata=True
txed                            0.1 GB  metadata=True
vcseis                 

In [ ]:
import seisbench.data as sbd

cached_with_meta = [
    'STEAD', 'NEIC', 'GEOFON', 'ETHZ', 'SCEDC', 
    'Iquique', 'LenDB', 'PNW', 'OBST2024', 'TXED', 
    'VCSEIS', 'InstanceCounts', 'CEED',
]

for name in cached_with_meta:
    try:
        cls = getattr(sbd, name)
        ds = cls()  # loads from cache, no download
        cols = ds.metadata.columns.tolist()
        
        # Search for P and S pick columns (naming varies across datasets)
        p_cols = [c for c in cols if 'p_' in c.lower() and ('arrival' in c.lower() or 'pick' in c.lower())]
        s_cols = [c for c in cols if 's_' in c.lower() and ('arrival' in c.lower() or 'pick' in c.lower())]
        
        print(f"\n{'='*60}")
        print(f"{name}  |  {len(ds):,} traces")
        print(f"  P pick columns: {p_cols}")
        print(f"  S pick columns: {s_cols}")
        print(f"  All columns: {cols}")
    except Exception as e:
        print(f"\n{name}: FAILED — {e}")


STEAD: FAILED — Found partial instance. This suggests that either the download is currently in progress or a download failed. To redownload the file, call the dataset with force=True. To wait for another download to finish, use wait_for_file=True.

NEIC: FAILED — Found partial instance. This suggests that either the download is currently in progress or a download failed. To redownload the file, call the dataset with force=True. To wait for another download to finish, use wait_for_file=True.


2026-05-05 16:40:01,796 | seisbench | WARNING | Dataset GEOFON not in cache.
2026-05-05 16:40:01,798 | seisbench | WARNING | Trying to download preprocessed version from SeisBench repository.




KeyboardInterrupt



In [8]:
import pandas as pd
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"

cached_datasets = [
    'stead', 'neic', 'geofon', 'ethz', 'scedc',
    'iquique', 'lendb', 'pnw', 'obst2024', 'txed',
    'vcseis', 'instancecounts', 'ceed',
]

for name in cached_datasets:
    meta_path = cache / name / "metadata.csv"
    if not meta_path.exists():
        print(f"\n{name}: no metadata.csv found")
        continue
    
    df = pd.read_csv(meta_path, nrows=5)  # just 5 rows to see columns
    cols = df.columns.tolist()
    
    # Find anything that looks like a phase pick
    pick_cols = [c for c in cols if any(x in c.lower() for x in ['arrival', 'pick', 'p_', 's_', 'phase'])]
    
    print(f"\n{'='*60}")
    print(f"{name}  |  columns with pick/arrival/phase:")
    for c in pick_cols:
        print(f"  {c}: {df[c].iloc[:3].tolist()}")


stead  |  columns with pick/arrival/phase:
  trace_p_arrival_sample: [nan, nan, nan]
  trace_p_status: [nan, nan, nan]
  trace_p_weight: [nan, nan, nan]
  path_p_travel_sec: [nan, nan, nan]
  trace_s_arrival_sample: [nan, nan, nan]
  trace_s_status: [nan, nan, nan]
  trace_s_weight: [nan, nan, nan]
  source_gap_deg: [nan, nan, nan]
  source_mechanism_strike_dip_rake: [nan, nan, nan]

neic  |  columns with pick/arrival/phase:
  trace_p_arrival_sample: [1200.0, 1200.0, 1200.0]
  trace_p_status: ['manual', 'manual', 'manual']
  path_ep_distance_km: [27.7987316611, 73.3886515854, 18.9031375296]
  trace_s_arrival_sample: [nan, nan, nan]
  trace_s_status: [nan, nan, nan]

geofon  |  columns with pick/arrival/phase:
  station_sensitivity_counts_spm: [313600000.0, 3885600000.0, 629145000.0]
  trace_has_spikes: [False, False, False]
  trace_P_arrival_sample: [1200.30122, 1199.75908, 1199.60244]
  trace_P_status: ['manual', 'manual', 'manual']
  trace_pP_arrival_sample: [nan, 1477.34942, nan]
 

In [9]:
import pandas as pd
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"

# Column mapping — the key deliverable from this step
col_map = {
    'stead':          {'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'neic':           {'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'geofon':         {'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'ethz':           {'p': 'trace_P1_arrival_sample','s': 'trace_S1_arrival_sample'},
    'scedc':          {'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'iquique':        {'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'lendb':          {'p': 'trace_p_arrival_sample', 's': None},
    'pnw':            {'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'obst2024':       {'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'instancecounts': {'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'txed':           {'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
}

rows = []
for name, pcols in col_map.items():
    meta_path = cache / name / "metadata.csv"
    if not meta_path.exists():
        continue
    
    df = pd.read_csv(meta_path, usecols=[c for c in pcols.values() if c is not None])
    n_total = len(df)
    n_with_p = df[pcols['p']].notna().sum() if pcols['p'] else 0
    n_with_s = df[pcols['s']].notna().sum() if pcols['s'] else 0
    n_both = 0
    if pcols['p'] and pcols['s']:
        n_both = (df[pcols['p']].notna() & df[pcols['s']].notna()).sum()
    
    rows.append({
        'dataset': name,
        'total_traces': n_total,
        'has_P': n_with_p,
        'has_S': n_with_s,
        'has_both': n_both,
        'pct_both': round(100 * n_both / n_total, 1) if n_total > 0 else 0,
    })
    print(f"{name:18s}  total={n_total:>8,}  P={n_with_p:>8,}  S={n_with_s:>8,}  both={n_both:>8,}  ({100*n_both/n_total:.1f}%)")

summary = pd.DataFrame(rows)
summary.to_csv('benchmark_pool_summary.csv', index=False)
print("\nSaved to benchmark_pool_summary.csv")

stead               total=1,265,657  P=1,030,231  S=1,030,231  both=1,030,231  (81.4%)
neic                total=1,354,789  P=1,025,000  S= 329,789  both=       0  (0.0%)
geofon              total= 275,274  P= 274,474  S=   2,648  both=   2,583  (0.9%)
ethz                total=  36,743  P=   1,674  S=      86  both=      57  (0.2%)
scedc               total=8,035,833  P=7,501,488  S=4,317,447  both=3,783,102  (47.1%)
iquique             total=  13,400  P=  13,327  S=  11,361  both=  11,288  (84.2%)
lendb               total=1,244,942  P= 629,095  S=       0  both=       0  (0.0%)
pnw                 total= 183,909  P= 183,909  S= 183,909  both= 183,909  (100.0%)
obst2024            total=  60,394  P=  35,394  S=  35,394  both=  35,394  (58.6%)
instancecounts      total=1,159,249  P=1,159,249  S= 713,883  both= 713,883  (61.6%)
txed                total= 519,689  P= 312,231  S= 312,231  both= 312,231  (60.1%)

Saved to benchmark_pool_summary.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

cache = Path.home() / ".seisbench" / "datasets"

# Proposed benchmark split
config = {
    'stead':          {'n': 1000, 'category': 'Global mixed'},
    'scedc':          {'n': 800,  'category': 'California transform'},
    'instancecounts': {'n': 800,  'category': 'Italy subduction'},
    'pnw':            {'n': 800,  'category': 'Cascadia subduction'},
    'txed':           {'n': 600,  'category': 'Induced seismicity'},
    'obst2024':       {'n': 600,  'category': 'Ocean bottom'},
    'iquique':        {'n': 400,  'category': 'Chile aftershocks'},
    'neic':           {'n': 800,  'category': 'Teleseismic'},
}

def find_col(cols, candidates):
    """Find first matching column from a list of candidates."""
    for c in candidates:
        if c in cols:
            return c
    # Case-insensitive fallback
    cols_lower = {x.lower(): x for x in cols}
    for c in candidates:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None

def find_arrival_cols(cols):
    """Find P and S arrival sample columns using SeisBench convention."""
    p_col, s_col = None, None
    for c in cols:
        if c.endswith('_arrival_sample'):
            phase = c.replace('trace_', '').replace('_arrival_sample', '')
            if phase.lower() in ['p', 'p1', 'pg']:
                if p_col is None:  # take first match
                    p_col = c
            elif phase.lower() in ['s', 's1', 'sg']:
                if s_col is None:
                    s_col = c
    return p_col, s_col

all_samples = []

for ds_name, cfg in config.items():
    meta_path = cache / ds_name / "metadata.csv"
    if not meta_path.exists():
        print(f"SKIP {ds_name}: no metadata")
        continue
    
    df = pd.read_csv(meta_path, low_memory=False)
    cols = df.columns.tolist()
    
    # Find arrival columns
    p_col, s_col = find_arrival_cols(cols)
    
    # Filter to traces with P picks (and S if available)
    if ds_name == 'neic':
        # Teleseismic: P only is fine
        mask = df[p_col].notna() if p_col else pd.Series([False]*len(df))
    else:
        if p_col and s_col:
            mask = df[p_col].notna() & df[s_col].notna()
        elif p_col:
            mask = df[p_col].notna()
        else:
            print(f"SKIP {ds_name}: no arrival columns found")
            continue
    
    pool = df[mask]
    n_sample = min(cfg['n'], len(pool))
    sample = pool.sample(n=n_sample, random_state=42)
    
    # Find standard columns
    mag_col = find_col(cols, ['source_magnitude', 'source_magnitude_value'])
    depth_col = find_col(cols, ['source_depth_km'])
    dist_col = find_col(cols, ['path_ep_distance_km', 'path_hyp_distance_km', 
                                'source_distance_km', 'path_ep_distance_deg'])
    sr_col = find_col(cols, ['trace_sampling_rate_hz'])
    npts_col = find_col(cols, ['trace_npts', 'trace_number_of_samples'])
    
    for _, row in sample.iterrows():
        entry = {
            'dataset': ds_name,
            'category': cfg['category'],
            'magnitude': row.get(mag_col) if mag_col else np.nan,
            'depth_km': row.get(depth_col) if depth_col else np.nan,
            'distance_km': row.get(dist_col) if dist_col else np.nan,
            'p_arrival_sample': row.get(p_col) if p_col else np.nan,
            's_arrival_sample': row.get(s_col) if s_col else np.nan,
            'sampling_rate': row.get(sr_col) if sr_col else np.nan,
        }
        all_samples.append(entry)
    
    print(f"OK  {ds_name:18s}  sampled {n_sample} from {len(pool):,} pool  "
          f"p_col={p_col}  s_col={s_col}  mag_col={mag_col}  dist_col={dist_col}")

bench = pd.DataFrame(all_samples)
print(f"\nTotal benchmark traces: {len(bench)}")

# Save the full benchmark metadata
bench.to_csv('benchmark_sample_metadata.csv', index=False)

# Print compact stats for visualization
stats = {}

# 1. Per-category counts
stats['category_counts'] = bench.groupby('category').size().to_dict()

# 2. Magnitude distribution (binned)
mag_valid = bench['magnitude'].dropna()
if len(mag_valid) > 0:
    bins = [-1, 0, 1, 2, 3, 4, 5, 6, 7, 10]
    labels = ['<0', '0-1', '1-2', '2-3', '3-4', '4-5', '5-6', '6-7', '7+']
    mag_binned = pd.cut(mag_valid, bins=bins, labels=labels)
    stats['magnitude_hist'] = mag_binned.value_counts().sort_index().to_dict()

# 3. Depth distribution (binned)
depth_valid = bench['depth_km'].dropna()
if len(depth_valid) > 0:
    bins = [0, 5, 10, 20, 50, 100, 200, 700]
    labels = ['0-5', '5-10', '10-20', '20-50', '50-100', '100-200', '200+']
    depth_binned = pd.cut(depth_valid.clip(lower=0), bins=bins, labels=labels)
    stats['depth_hist'] = depth_binned.value_counts().sort_index().to_dict()

# 4. Distance distribution (binned)
dist_valid = bench['distance_km'].dropna()
if len(dist_valid) > 0:
    bins = [0, 10, 30, 100, 300, 1000, 20000]
    labels = ['0-10', '10-30', '30-100', '100-300', '300-1000', '1000+']
    dist_binned = pd.cut(dist_valid, bins=bins, labels=labels)
    stats['distance_hist'] = dist_binned.value_counts().sort_index().to_dict()

# 5. P arrival position in trace (as fraction of trace length, binned)
if bench['p_arrival_sample'].notna().any() and bench['sampling_rate'].notna().any():
    p_time = bench['p_arrival_sample'] / bench['sampling_rate']
    p_time_valid = p_time.dropna()
    stats['p_arrival_seconds'] = {
        'mean': round(p_time_valid.mean(), 1),
        'median': round(p_time_valid.median(), 1),
        'std': round(p_time_valid.std(), 1),
        'min': round(p_time_valid.min(), 1),
        'max': round(p_time_valid.max(), 1),
    }

# 6. S-P time (seconds)
if bench['s_arrival_sample'].notna().any() and bench['p_arrival_sample'].notna().any():
    sp_time = (bench['s_arrival_sample'] - bench['p_arrival_sample']) / bench['sampling_rate']
    sp_valid = sp_time.dropna()
    sp_valid = sp_valid[sp_valid > 0]
    if len(sp_valid) > 0:
        bins = [0, 2, 5, 10, 20, 50, 100, 500]
        labels = ['0-2s', '2-5s', '5-10s', '10-20s', '20-50s', '50-100s', '100s+']
        sp_binned = pd.cut(sp_valid, bins=bins, labels=labels)
        stats['sp_time_hist'] = sp_binned.value_counts().sort_index().to_dict()

# 7. Per-category magnitude stats
mag_by_cat = bench.groupby('category')['magnitude'].describe().round(2)
stats['mag_by_category'] = mag_by_cat.to_dict()

# 8. Per-category depth stats  
depth_by_cat = bench.groupby('category')['depth_km'].describe().round(2)
stats['depth_by_category'] = depth_by_cat.to_dict()

# 9. Missing data summary
stats['missing'] = {
    'magnitude': int(bench['magnitude'].isna().sum()),
    'depth': int(bench['depth_km'].isna().sum()),
    'distance': int(bench['distance_km'].isna().sum()),
    'p_arrival': int(bench['p_arrival_sample'].isna().sum()),
    's_arrival': int(bench['s_arrival_sample'].isna().sum()),
}

print("\n" + "="*60)
print("STATS FOR VISUALIZATION:")
print("="*60)
print(json.dumps(stats, indent=2, default=str))

OK  stead               sampled 1000 from 1,030,231 pool  p_col=trace_p_arrival_sample  s_col=trace_s_arrival_sample  mag_col=source_magnitude  dist_col=source_distance_km


In [1]:
import pandas as pd
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"

datasets = ['stead', 'scedc', 'instancecounts', 'pnw', 'txed', 
            'obst2024', 'iquique', 'neic', 'geofon', 'lendb']

for name in datasets:
    meta_path = cache / name / "metadata.csv"
    if not meta_path.exists():
        continue
    
    df = pd.read_csv(meta_path, nrows=1000, low_memory=False)
    
    # Find the columns we care about
    cols = df.columns.tolist()
    mag = [c for c in cols if 'magnitude' in c.lower() and 'type' not in c.lower()]
    depth = [c for c in cols if 'depth' in c.lower()]
    dist = [c for c in cols if 'distance' in c.lower()]
    sr = [c for c in cols if 'sampling_rate' in c.lower()]
    arr = [c for c in cols if 'arrival_sample' in c.lower()]
    
    print(f"\n{name}")
    print(f"  magnitude: {mag}")
    print(f"  depth:     {depth}")
    print(f"  distance:  {dist}")
    print(f"  samp_rate: {sr}")
    print(f"  arrivals:  {arr}")


stead
  magnitude: ['source_magnitude', 'source_magnitude_author']
  depth:     ['source_depth_km', 'source_depth_uncertainty_km']
  distance:  ['source_distance_deg', 'source_distance_km']
  samp_rate: []
  arrivals:  ['trace_p_arrival_sample', 'trace_s_arrival_sample']

scedc
  magnitude: ['source_magnitude']
  depth:     ['source_depth_km']
  distance:  ['station_epicentral_distance']
  samp_rate: ['trace_sampling_rate_hz']
  arrivals:  ['trace_p_arrival_sample', 'trace_s_arrival_sample']

instancecounts
  magnitude: ['source_magnitude']
  depth:     ['source_depth_km', 'source_depth_uncertainty_km']
  distance:  ['path_ep_distance_km', 'path_hyp_distance_km']
  samp_rate: []
  arrivals:  ['trace_P_arrival_sample', 'trace_S_arrival_sample']

pnw
  magnitude: ['preferred_source_magnitude', 'preferred_source_magnitude_uncertainty', 'source_local_magnitude', 'source_local_magnitude_uncertainty', 'source_duration_magnitude', 'source_duration_magnitude_uncertainty', 'source_hand_magnitu

In [2]:
import pandas as pd
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"

for name in ['pnw', 'txed', 'iquique', 'geofon']:
    df = pd.read_csv(cache / name / "metadata.csv", nrows=5)
    geo = [c for c in df.columns if any(x in c.lower() for x in 
           ['latitude', 'longitude', 'lat_', 'lon_', '_lat', '_lon'])]
    print(f"{name}: {geo}")

pnw: ['source_latitude_deg', 'source_longitude_deg', 'station_latitude_deg', 'station_longitude_deg']
txed: ['source_latitude_deg', 'source_longitude_deg', 'station_latitude_deg', 'station_longitude_deg']
iquique: ['source_latitude_deg', 'source_longitude_deg', 'station_latitude_deg', 'station_longitude_deg']
geofon: ['source_latitude_deg', 'source_latitude_uncertainty_deg', 'source_longitude_deg', 'source_longitude_uncertainty_deg', 'station_latitude_deg', 'station_longitude_deg']


In [3]:
import seisbench.data as sbd

check = ['OBS', 'MLAAPDE', 'ISC_EHB_DepthPhases', 'VCSEIS', 'CWA', 
         'CREW', 'CEED', 'Ross2018GPD', 'Ross2018JGRPick', 
         'Meier2019JGR', 'AQ2009Counts', 'PNWExotic', 'PNWNoise',
         'CWANoise', 'InstanceNoise', 'InstanceCountsCombined',
         'PNWAccelerometers', 'LFEStacksCascadiaBostock2015']

for name in check:
    try:
        cls = getattr(sbd, name)
        doc = (cls.__doc__ or "No docstring").strip().split('\n')
        # Print first 5 lines of docstring
        print(f"\n{'='*60}")
        print(f"{name}")
        for line in doc[:8]:
            print(f"  {line.strip()}")
    except Exception as e:
        print(f"\n{name}: {e}")


OBS
  OBS Benchmark Dataset of local events
  
  Default component order is 'Z12H'. You can easily omit one component like, e.g., hydrophone by explicitly passing
  parameter 'component_order="Z12"'. This way, the dataset can be input to land station pickers that use only 3
  components.

MLAAPDE
  MLAAPDE dataset from Cole et al. (2023)
  
  Note that the SeisBench version is not identical to the precompiled version
  distributed directly through USGS but uses a different data selection.
  In addition, custom versions of MLAAPDE can be compiled with the software
  provided by the original authors. These datasets can be exported in
  SeisBench format.

ISC_EHB_DepthPhases
  Dataset of depth phase picks from the
  `ISC-EHB bulletin <http://www.isc.ac.uk/isc-ehb/>`_.

VCSEIS
  A data set of seismic waveforms from various volcanic regions: Alaska, Hawaii, Northern California, Cascade volcanoes.

CWA
  CWA dataset - Events and traces.

CREW
  Curated Regional Earthquake Waveforms (CREW da

In [4]:
import seisbench.data as sbd

# Check if these have a citation or size hint before downloading
for name in ['MLAAPDE', 'CWA', 'VCSEIS']:
    cls = getattr(sbd, name)
    print(f"\n{name}")
    if hasattr(cls, '_citation'):
        print(f"  citation: {cls._citation}")
    # Try to see what path it would download to
    print(f"  cache path: {cls._path()}")


MLAAPDE


AttributeError: type object 'MLAAPDE' has no attribute '_path'

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"

col_map = {
    'stead':          {'mag': 'source_magnitude',            'depth': 'source_depth_km', 'dist': 'source_distance_km',       'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'scedc':          {'mag': 'source_magnitude',            'depth': 'source_depth_km', 'dist': 'station_epicentral_distance','p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'instancecounts': {'mag': 'source_magnitude',            'depth': 'source_depth_km', 'dist': 'path_ep_distance_km',       'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'pnw':            {'mag': 'preferred_source_magnitude',  'depth': 'source_depth_km', 'dist': None,                        'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'txed':           {'mag': 'source_magnitude',            'depth': 'source_depth_km', 'dist': None,                        'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'obst2024':       {'mag': 'source_magnitude',            'depth': 'source_depth_km', 'dist': 'source_distance_km',        'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'iquique':        {'mag': None,                          'depth': 'source_depth_km', 'dist': None,                        'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'neic':           {'mag': 'source_magnitude',            'depth': None,              'dist': 'path_ep_distance_km',        'p': 'trace_p_arrival_sample', 's': 'trace_s_arrival_sample'},
    'geofon':         {'mag': 'source_magnitude',            'depth': 'source_depth_km', 'dist': None,                        'p': 'trace_P_arrival_sample', 's': 'trace_S_arrival_sample'},
    'lendb':          {'mag': 'source_magnitude',            'depth': 'source_depth_km', 'dist': 'path_ep_distance_km',        'p': 'trace_p_arrival_sample', 's': None},
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

frames = []
for name, cmap in col_map.items():
    meta_path = cache / name / "metadata.csv"
    if not meta_path.exists():
        print(f"SKIP {name}")
        continue
    
    # Figure out which columns to load
    load_cols = [c for c in cmap.values() if c is not None]
    if cmap['dist'] is None:
        load_cols += ['source_latitude_deg', 'source_longitude_deg',
                      'station_latitude_deg', 'station_longitude_deg']
    
    df = pd.read_csv(meta_path, usecols=load_cols, low_memory=False)
    
    # Sample 10k
    if len(df) > 10000:
        df = df.sample(10000, random_state=42)
    
    # Build standardized columns
    out = pd.DataFrame()
    out['dataset'] = name
    out['magnitude'] = df[cmap['mag']].values if cmap['mag'] else np.nan
    out['depth_km'] = df[cmap['depth']].values if cmap['depth'] else np.nan
    out['p_sample'] = df[cmap['p']].values if cmap['p'] else np.nan
    out['s_sample'] = df[cmap['s']].values if cmap['s'] else np.nan
    
    # Distance: use existing column or compute from lat/lon
    if cmap['dist']:
        out['distance_km'] = df[cmap['dist']].values
    else:
        out['distance_km'] = haversine_km(
            df['source_latitude_deg'].values, df['source_longitude_deg'].values,
            df['station_latitude_deg'].values, df['station_longitude_deg'].values
        )
    
    frames.append(out)
    print(f"OK  {name:18s}  {len(out)} rows")

pool = pd.concat(frames, ignore_index=True)
pool.to_csv('benchmark_pool_10k_sample.csv', index=False)
print(f"\nTotal: {len(pool)} rows")
print(f"Saved to benchmark_pool_10k_sample.csv")

# Quick summary stats
print(f"\nMagnitude: {pool['magnitude'].describe().round(2).to_dict()}")
print(f"Depth:     {pool['depth_km'].describe().round(2).to_dict()}")
print(f"Distance:  {pool['distance_km'].describe().round(2).to_dict()}")

OK  stead               10000 rows



KeyboardInterrupt



In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"

col_map = {
    'stead':          {'mag': 'source_magnitude', 'depth': 'source_depth_km', 'dist': 'source_distance_km'},
    'scedc':          {'mag': 'source_magnitude', 'depth': 'source_depth_km', 'dist': 'station_epicentral_distance'},
    'instancecounts': {'mag': 'source_magnitude', 'depth': 'source_depth_km', 'dist': 'path_ep_distance_km'},
    'pnw':            {'mag': 'preferred_source_magnitude', 'depth': 'source_depth_km', 'dist': None},
    'txed':           {'mag': 'source_magnitude', 'depth': 'source_depth_km', 'dist': None},
    'obst2024':       {'mag': 'source_magnitude', 'depth': 'source_depth_km', 'dist': 'source_distance_km'},
    'iquique':        {'mag': None,               'depth': 'source_depth_km', 'dist': None},
    'neic':           {'mag': 'source_magnitude', 'depth': None,              'dist': 'path_ep_distance_km'},
    'geofon':         {'mag': 'source_magnitude', 'depth': 'source_depth_km', 'dist': None},
    'lendb':          {'mag': 'source_magnitude', 'depth': 'source_depth_km', 'dist': 'path_ep_distance_km'},
}

# Process one dataset at a time using chunked reading for large ones
mag_bins = [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 10]
depth_bins = [0, 5, 10, 20, 35, 70, 150, 300, 700]
dist_bins = [0, 10, 30, 100, 300, 1000, 5000, 20000]

results = {}

for name, cmap in col_map.items():
    meta_path = cache / name / "metadata.csv"
    if not meta_path.exists():
        print(f"SKIP {name}")
        continue
    
    usecols = [c for c in cmap.values() if c is not None]
    
    mag_counts = np.zeros(len(mag_bins)-1)
    depth_counts = np.zeros(len(depth_bins)-1)
    dist_counts = np.zeros(len(dist_bins)-1)
    total = 0
    
    for chunk in pd.read_csv(meta_path, usecols=usecols, chunksize=100000, low_memory=False):
        total += len(chunk)
        
        if cmap['mag'] and cmap['mag'] in chunk.columns:
            vals = chunk[cmap['mag']].dropna().values
            mag_counts += np.histogram(vals, bins=mag_bins)[0]
        
        if cmap['depth'] and cmap['depth'] in chunk.columns:
            vals = chunk[cmap['depth']].dropna().clip(lower=0).values
            depth_counts += np.histogram(vals, bins=depth_bins)[0]
        
        if cmap['dist'] and cmap['dist'] in chunk.columns:
            vals = chunk[cmap['dist']].dropna().values
            dist_counts += np.histogram(vals, bins=dist_bins)[0]
    
    results[name] = {
        'total': total,
        'mag': mag_counts.astype(int).tolist(),
        'depth': depth_counts.astype(int).tolist(),
        'dist': dist_counts.astype(int).tolist(),
    }
    print(f"OK {name:18s} {total:>10,} traces")

import json
print("\n" + json.dumps({
    'mag_bins': mag_bins,
    'depth_bins': depth_bins,
    'dist_bins': dist_bins,
    'datasets': results
}, indent=2))

OK stead               1,265,657 traces
OK scedc               8,035,833 traces
OK instancecounts      1,159,249 traces
OK pnw                   183,909 traces
OK txed                  519,689 traces
OK obst2024               60,394 traces
OK iquique                13,400 traces
OK neic                1,354,789 traces
OK geofon                275,274 traces
OK lendb               1,244,942 traces

{
  "mag_bins": [
    -2,
    -1,
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    10
  ],
  "depth_bins": [
    0,
    5,
    10,
    20,
    35,
    70,
    150,
    300,
    700
  ],
  "dist_bins": [
    0,
    10,
    30,
    100,
    300,
    1000,
    5000,
    20000
  ],
  "datasets": {
    "stead": {
      "total": 1265657,
      "mag": [
        0,
        242,
        347189,
        411465,
        167233,
        80156,
        21911,
        1889,
        143,
        3
      ],
      "depth": [
        305010,
        294644,
        276853,
        43948,
        561

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Compute distance for datasets missing it
for name in ['pnw', 'txed', 'iquique', 'geofon']:
    meta_path = cache / name / "metadata.csv"
    df = pd.read_csv(meta_path, usecols=['source_latitude_deg','source_longitude_deg',
                                          'station_latitude_deg','station_longitude_deg'],
                     low_memory=False)
    dist = haversine_km(df['source_latitude_deg'].values, df['source_longitude_deg'].values,
                        df['station_latitude_deg'].values, df['station_longitude_deg'].values)
    
    dist_valid = dist[~np.isnan(dist)]
    bins = [0, 10, 30, 100, 300, 1000, 5000, 20000]
    hist = np.histogram(dist_valid, bins=bins)[0]
    print(f"{name:12s}  valid={len(dist_valid):>8,}  bins={hist.tolist()}")

# Check OBST2024 — what columns actually have data?
print("\nOBST2024 non-null columns:")
df = pd.read_csv(cache / "obst2024" / "metadata.csv", nrows=100)
for c in df.columns:
    n = df[c].notna().sum()
    if n > 0:
        print(f"  {c}: {n}/100 non-null  ex: {df[c].dropna().iloc[:2].tolist()}")

pnw           valid= 183,909  bins=[26087, 64669, 83718, 9206, 229, 0, 0]
txed          valid= 312,231  bins=[26131, 74327, 123689, 87680, 404, 0, 0]
iquique       valid=  13,400  bins=[35, 313, 3894, 8562, 596, 0, 0]
geofon        valid= 269,500  bins=[15, 87, 830, 4737, 14907, 79076, 169848]

OBST2024 non-null columns:
  station_network_code: 100/100 non-null  ex: ['ZD', 'ZD']
  station_code: 100/100 non-null  ex: ['G16', 'D08']
  station_latitude_deg: 100/100 non-null  ex: [-4.7169, -3.9513]
  station_longitude_deg: 100/100 non-null  ex: [-105.584198, -104.351303]
  station_elevation_m: 100/100 non-null  ex: [-2961.0, -2681.0]
  trace_p_arrival_sample: 100/100 non-null  ex: [1523.0, 804.0]
  trace_s_arrival_sample: 100/100 non-null  ex: [1785.0, 1020.0]
  trace_snr_db: 100/100 non-null  ex: ['[18.388 14.578 15.233]', '[4.604 4.249 4.853]']
  trace_coda_end_sample: 100/100 non-null  ex: [1837.0, 1063.0]
  trace_start_time: 100/100 non-null  ex: ['2008-07-09T01:09:24.670400Z', '2008-0

In [11]:
import pandas as pd
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"
df = pd.read_csv(cache / "obst2024" / "metadata.csv", nrows=5)
src = [c for c in df.columns if 'source' in c.lower()]
print(f"Source columns: {src}")

Source columns: ['source_id', 'source_origin_time', 'source_origin_uncertainty_sec', 'source_latitude_deg', 'source_longitude_deg', 'source_error_sec', 'source_gap_deg', 'source_horizontal_uncertainty_km', 'source_depth_km', 'source_depth_uncertainty_km', 'source_magnitude', 'source_magnitude_type', 'source_magnitude_author', 'source_mechanism_strike_dip_rake', 'source_distance_deg', 'source_distance_km']


In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

cache = Path.home() / ".seisbench" / "datasets"
df = pd.read_csv(cache / "obst2024" / "metadata.csv",
                 usecols=['source_magnitude','source_depth_km','source_distance_km'],
                 low_memory=False)

print(f"Total rows: {len(df)}")
print(f"Magnitude non-null: {df['source_magnitude'].notna().sum()}")
print(f"Depth non-null:     {df['source_depth_km'].notna().sum()}")
print(f"Distance non-null:  {df['source_distance_km'].notna().sum()}")

mag_bins = [-2,-1,0,1,2,3,4,5,6,7,10]
depth_bins = [0,5,10,20,35,70,150,300,700]
dist_bins = [0,10,30,100,300,1000,5000,20000]

mag_v = df['source_magnitude'].dropna().values
dep_v = df['source_depth_km'].dropna().clip(lower=0).values
dis_v = df['source_distance_km'].dropna().values

print(f"\nMagnitude: {np.histogram(mag_v, bins=mag_bins)[0].tolist()}")
print(f"Depth:     {np.histogram(dep_v, bins=depth_bins)[0].tolist()}")
print(f"Distance:  {np.histogram(dis_v, bins=dist_bins)[0].tolist()}")

Total rows: 60394
Magnitude non-null: 0
Depth non-null:     0
Distance non-null:  0

Magnitude: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Depth:     [0, 0, 0, 0, 0, 0, 0, 0]
Distance:  [0, 0, 0, 0, 0, 0, 0]


In [17]:
"""
Build a physics-stratified benchmark dataset for evaluating PhaseNet generalization.

Stratification axes (seismologically motivated):
  Distance: 0-150 km (local), 150-1500 km (regional), >1500 km (teleseismic)
  Depth:    0-15 km (upper crustal), 15-70 km (lower crust), 70-300 km (intermediate), 300+ km (deep focus)
  Magnitude: <3 (micro), 3-6 (moderate), 6+ (large)

Each trace retains its source dataset and tectonic type so that during evaluation,
traces from the model's training dataset can be excluded (fair cross-domain testing).

Datasets with partial metadata (NEIC: no depth, Iquique: no magnitude) are binned
on their available axes and sampled separately to ensure representation.

Usage:
  Run this script, or copy cells into a Jupyter notebook.
  Requires: pandas, numpy
  Input: SeisBench cached metadata CSVs in ~/.seisbench/datasets/
  Output: benchmark_manifest.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# Configuration
# ============================================================

cache = Path.home() / ".seisbench" / "datasets"

TARGET_PER_BIN = 150          # traces per physics bin (drawn across datasets)
MAX_PER_DATASET_PER_BIN = 80  # cap per dataset within a bin to ensure diversity
OBST2024_FLAT_SAMPLE = 400    # OBST2024 has no mag/depth/dist — flat sample
NEIC_TARGET_PER_BIN = 150     # NEIC: distance × magnitude bins
IQUIQUE_TARGET_PER_BIN = 80   # Iquique: distance × depth bins
RANDOM_SEED = 42

# Distance bins (km)
DIST_EDGES = [0, 150, 1500, np.inf]
DIST_LABELS = ['local (<150km)', 'regional (150-1500km)', 'teleseismic (>1500km)']

# Depth bins (km)
DEPTH_EDGES = [0, 15, 70, 300, np.inf]
DEPTH_LABELS = ['upper-crustal (0-15km)', 'lower-crust (15-70km)', 
                'intermediate (70-300km)', 'deep-focus (>300km)']

# Magnitude bins
MAG_EDGES = [-np.inf, 3.0, 6.0, np.inf]
MAG_LABELS = ['micro (M<3)', 'moderate (M3-6)', 'large (M>6)']

# Datasets with FULL physics metadata (distance + depth + magnitude)
FULL_DATASETS = {
    'stead': {
        'mag': 'source_magnitude',
        'depth': 'source_depth_km',
        'dist': 'source_distance_km',
        'p': 'trace_p_arrival_sample',
        's': 'trace_s_arrival_sample',
        'compute_dist': False,
        'tectonic_type': 'global_mixed',
        'region': 'Global (US-heavy)',
        'trained_models': ['stead'],
    },
    'scedc': {
        'mag': 'source_magnitude',
        'depth': 'source_depth_km',
        'dist': 'station_epicentral_distance',
        'p': 'trace_p_arrival_sample',
        's': 'trace_s_arrival_sample',
        'compute_dist': False,
        'tectonic_type': 'transform',
        'region': 'Southern California',
        'trained_models': ['scedc'],
    },
    'instancecounts': {
        'mag': 'source_magnitude',
        'depth': 'source_depth_km',
        'dist': 'path_ep_distance_km',
        'p': 'trace_P_arrival_sample',
        's': 'trace_S_arrival_sample',
        'compute_dist': False,
        'tectonic_type': 'subduction_crustal',
        'region': 'Italy / Mediterranean',
        'trained_models': ['instance'],
    },
    'pnw': {
        'mag': 'preferred_source_magnitude',
        'depth': 'source_depth_km',
        'dist': None,
        'p': 'trace_P_arrival_sample',
        's': 'trace_S_arrival_sample',
        'compute_dist': True,
        'tectonic_type': 'subduction',
        'region': 'Pacific Northwest / Cascadia',
        'trained_models': [],
    },
    'txed': {
        'mag': 'source_magnitude',
        'depth': 'source_depth_km',
        'dist': None,
        'p': 'trace_p_arrival_sample',
        's': 'trace_s_arrival_sample',
        'compute_dist': True,
        'tectonic_type': 'induced',
        'region': 'Texas (induced seismicity)',
        'trained_models': [],
    },
    'geofon': {
        'mag': 'source_magnitude',
        'depth': 'source_depth_km',
        'dist': None,
        'p': 'trace_P_arrival_sample',
        's': 'trace_S_arrival_sample',
        'compute_dist': True,
        'tectonic_type': 'global_broadband',
        'region': 'Global (GEOFON broadband)',
        'trained_models': ['geofon'],
        'p_only': True,
    },
    'lendb': {
        'mag': 'source_magnitude',
        'depth': 'source_depth_km',
        'dist': 'path_ep_distance_km',
        'p': 'trace_p_arrival_sample',
        's': None,
        'p_only': True,
        'compute_dist': False,
        'tectonic_type': 'regional_mixed',
        'region': 'Central Italy',
        'trained_models': ['lendb'],
    },
}

# NEIC: has distance + magnitude but NO depth
NEIC_CONFIG = {
    'mag': 'source_magnitude',
    'dist': 'path_ep_distance_km',
    'p': 'trace_p_arrival_sample',
    's': 'trace_s_arrival_sample',
    'p_only': True,
    'tectonic_type': 'global_teleseismic',
    'region': 'Global (teleseismic)',
    'trained_models': ['neic'],
}

# Iquique: has distance + depth but NO magnitude
IQUIQUE_CONFIG = {
    'depth': 'source_depth_km',
    'p': 'trace_P_arrival_sample',
    's': 'trace_S_arrival_sample',
    'compute_dist': True,
    'tectonic_type': 'subduction_aftershock',
    'region': 'Northern Chile',
    'trained_models': ['iquique'],
}

# OBST2024: has P/S picks and SNR only — no mag/depth/dist
OBST2024_CONFIG = {
    'p': 'trace_p_arrival_sample',
    's': 'trace_s_arrival_sample',
    'tectonic_type': 'ocean_bottom',
    'region': 'East Pacific Rise (OBS)',
    'trained_models': ['obs'],
}


# ============================================================
# Helper functions
# ============================================================

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def assign_bins_vectorized(series, edges, labels):
    """Assign bins to a pandas Series. Returns Series of labels."""
    return pd.cut(series, bins=edges, labels=labels, right=False, include_lowest=True)


def read_dataset(meta_path, usecols, p_col, s_col, p_only, chunksize=500000):
    """Read a dataset in chunks, filtering to traces with valid P (and optionally S) picks."""
    chunks = []
    for chunk in pd.read_csv(meta_path, usecols=usecols, chunksize=chunksize, low_memory=False):
        mask = chunk[p_col].notna()
        if s_col and not p_only:
            mask = mask & chunk[s_col].notna()
        chunk = chunk[mask]
        if len(chunk) > 0:
            chunks.append(chunk)
    return pd.concat(chunks, ignore_index=False) if chunks else None


# ============================================================
# 1. Process FULL datasets (have all 3 physics axes)
# ============================================================

print("=" * 80)
print("PROCESSING FULL DATASETS (distance + depth + magnitude)")
print("=" * 80)

full_traces = []

for ds_name, cfg in FULL_DATASETS.items():
    meta_path = cache / ds_name / "metadata.csv"
    if not meta_path.exists():
        print(f"SKIP {ds_name}: metadata not found")
        continue
    
    # Build column list
    usecols = [cfg['p']]
    if cfg.get('s'):
        usecols.append(cfg['s'])
    if cfg['mag']:
        usecols.append(cfg['mag'])
    if cfg['depth']:
        usecols.append(cfg['depth'])
    if cfg.get('dist'):
        usecols.append(cfg['dist'])
    if cfg.get('compute_dist', False):
        usecols.extend(['source_latitude_deg', 'source_longitude_deg',
                        'station_latitude_deg', 'station_longitude_deg'])
    usecols = list(dict.fromkeys(usecols))
    
    print(f"Reading {ds_name}...", end=" ", flush=True)
    
    raw = read_dataset(meta_path, usecols, cfg['p'], cfg.get('s'), cfg.get('p_only', False))
    if raw is None:
        print("no valid traces")
        continue
    
    # Standardize
    out = pd.DataFrame()
    out['magnitude'] = raw[cfg['mag']].values
    out['depth_km'] = raw[cfg['depth']].values
    out['p_arrival_sample'] = raw[cfg['p']].values
    out['s_arrival_sample'] = raw[cfg['s']].values if cfg.get('s') else np.nan
    out['original_index'] = raw.index.values
    
    if cfg.get('compute_dist', False):
        out['distance_km'] = haversine_km(
            raw['source_latitude_deg'].values, raw['source_longitude_deg'].values,
            raw['station_latitude_deg'].values, raw['station_longitude_deg'].values
        )
    else:
        out['distance_km'] = raw[cfg['dist']].values
    
    # Assign 3-axis bins
    out['dist_bin'] = assign_bins_vectorized(out['distance_km'], DIST_EDGES, DIST_LABELS)
    out['depth_bin'] = assign_bins_vectorized(out['depth_km'], DEPTH_EDGES, DEPTH_LABELS)
    out['mag_bin'] = assign_bins_vectorized(out['magnitude'], MAG_EDGES, MAG_LABELS)
    
    # Composite bin — only for traces with ALL 3 axes
    fully_binned = out['dist_bin'].notna() & out['depth_bin'].notna() & out['mag_bin'].notna()
    out['physics_bin'] = np.where(
        fully_binned,
        out['dist_bin'].astype(str) + ' | ' + out['depth_bin'].astype(str) + ' | ' + out['mag_bin'].astype(str),
        'partial'
    )
    
    out['dataset'] = ds_name
    out['tectonic_type'] = cfg['tectonic_type']
    out['region'] = cfg['region']
    out['trained_models'] = ','.join(cfg['trained_models'])
    out['has_s_pick'] = out['s_arrival_sample'].notna()
    
    n_full = fully_binned.sum()
    full_traces.append(out[fully_binned])  # keep only fully-binned for main sampling
    print(f"{len(out):,} valid, {n_full:,} fully binned")


# ============================================================
# 2. Process NEIC separately (distance × magnitude, no depth)
# ============================================================

print(f"\n{'='*80}")
print("PROCESSING NEIC (distance × magnitude, depth unknown)")
print("=" * 80)

neic_path = cache / "neic" / "metadata.csv"
neic_traces = None
if neic_path.exists():
    cfg = NEIC_CONFIG
    usecols = [cfg['p'], cfg['s'], cfg['mag'], cfg['dist']]
    
    print("Reading neic...", end=" ", flush=True)
    raw = read_dataset(neic_path, usecols, cfg['p'], cfg.get('s'), cfg.get('p_only', True))
    
    if raw is not None:
        out = pd.DataFrame()
        out['magnitude'] = raw[cfg['mag']].values
        out['depth_km'] = np.nan
        out['distance_km'] = raw[cfg['dist']].values
        out['p_arrival_sample'] = raw[cfg['p']].values
        out['s_arrival_sample'] = raw[cfg['s']].values if cfg.get('s') else np.nan
        out['original_index'] = raw.index.values
        
        out['dist_bin'] = assign_bins_vectorized(out['distance_km'], DIST_EDGES, DIST_LABELS)
        out['depth_bin'] = np.nan
        out['mag_bin'] = assign_bins_vectorized(out['magnitude'], MAG_EDGES, MAG_LABELS)
        
        valid = out['dist_bin'].notna() & out['mag_bin'].notna()
        out['physics_bin'] = np.where(
            valid,
            out['dist_bin'].astype(str) + ' | depth_unknown | ' + out['mag_bin'].astype(str),
            'unusable'
        )
        
        out['dataset'] = 'neic'
        out['tectonic_type'] = cfg['tectonic_type']
        out['region'] = cfg['region']
        out['trained_models'] = ','.join(cfg['trained_models'])
        out['has_s_pick'] = out['s_arrival_sample'].notna()
        
        neic_traces = out[valid]
        print(f"{len(raw):,} valid, {valid.sum():,} binned (dist × mag)")


# ============================================================
# 3. Process Iquique separately (distance × depth, no magnitude)
# ============================================================

print(f"\n{'='*80}")
print("PROCESSING IQUIQUE (distance × depth, magnitude unknown)")
print("=" * 80)

iq_path = cache / "iquique" / "metadata.csv"
iq_traces = None
if iq_path.exists():
    cfg = IQUIQUE_CONFIG
    usecols = [cfg['p'], cfg['s'], cfg['depth'],
               'source_latitude_deg', 'source_longitude_deg',
               'station_latitude_deg', 'station_longitude_deg']
    
    print("Reading iquique...", end=" ", flush=True)
    raw = read_dataset(iq_path, usecols, cfg['p'], cfg.get('s'), False)
    
    if raw is not None:
        out = pd.DataFrame()
        out['magnitude'] = np.nan
        out['depth_km'] = raw[cfg['depth']].values
        out['distance_km'] = haversine_km(
            raw['source_latitude_deg'].values, raw['source_longitude_deg'].values,
            raw['station_latitude_deg'].values, raw['station_longitude_deg'].values
        )
        out['p_arrival_sample'] = raw[cfg['p']].values
        out['s_arrival_sample'] = raw[cfg['s']].values
        out['original_index'] = raw.index.values
        
        out['dist_bin'] = assign_bins_vectorized(out['distance_km'], DIST_EDGES, DIST_LABELS)
        out['depth_bin'] = assign_bins_vectorized(out['depth_km'], DEPTH_EDGES, DEPTH_LABELS)
        out['mag_bin'] = np.nan
        
        valid = out['dist_bin'].notna() & out['depth_bin'].notna()
        out['physics_bin'] = np.where(
            valid,
            out['dist_bin'].astype(str) + ' | ' + out['depth_bin'].astype(str) + ' | mag_unknown',
            'unusable'
        )
        
        out['dataset'] = 'iquique'
        out['tectonic_type'] = cfg['tectonic_type']
        out['region'] = cfg['region']
        out['trained_models'] = ','.join(cfg['trained_models'])
        out['has_s_pick'] = True
        
        iq_traces = out[valid]
        print(f"{len(raw):,} valid, {valid.sum():,} binned (dist × depth)")


# ============================================================
# 4. Process OBST2024 (flat sample, no physics metadata)
# ============================================================

print(f"\n{'='*80}")
print("PROCESSING OBST2024 (flat sample, no physics metadata)")
print("=" * 80)

obst_path = cache / "obst2024" / "metadata.csv"
obst_traces = None
if obst_path.exists():
    cfg = OBST2024_CONFIG
    df = pd.read_csv(obst_path, usecols=[cfg['p'], cfg['s']], low_memory=False)
    mask = df[cfg['p']].notna() & df[cfg['s']].notna()
    df = df[mask]
    
    out = pd.DataFrame()
    out['magnitude'] = np.nan
    out['depth_km'] = np.nan
    out['distance_km'] = np.nan
    out['p_arrival_sample'] = df[cfg['p']].values
    out['s_arrival_sample'] = df[cfg['s']].values
    out['original_index'] = df.index.values
    out['dist_bin'] = np.nan
    out['depth_bin'] = np.nan
    out['mag_bin'] = np.nan
    out['physics_bin'] = 'OBS_flat'
    out['dataset'] = 'obst2024'
    out['tectonic_type'] = cfg['tectonic_type']
    out['region'] = cfg['region']
    out['trained_models'] = ','.join(cfg['trained_models'])
    out['has_s_pick'] = True
    
    obst_traces = out
    print(f"{len(out):,} valid traces")


# ============================================================
# 5. Show bin populations before sampling
# ============================================================

# Combine all for reporting
all_pools = []
if full_traces:
    all_pools.append(pd.concat(full_traces, ignore_index=True))
if neic_traces is not None:
    all_pools.append(neic_traces)
if iq_traces is not None:
    all_pools.append(iq_traces)
if obst_traces is not None:
    all_pools.append(obst_traces)

pool = pd.concat(all_pools, ignore_index=True)

print(f"\n{'='*80}")
print(f"TOTAL POOL: {len(pool):,} traces")
print(f"{'='*80}")

bin_counts = pool.groupby('physics_bin').agg(
    total=('dataset', 'size'),
    n_datasets=('dataset', 'nunique'),
    datasets=('dataset', lambda x: ', '.join(sorted(x.unique())))
).sort_values('total', ascending=False)

print(f"\nPHYSICS BIN POPULATIONS:")
for idx, row in bin_counts.iterrows():
    print(f"  {idx:70s}  n={row['total']:>8,}  from {row['n_datasets']} ds: {row['datasets']}")

print(f"\nTotal populated bins: {len(bin_counts)}")


# ============================================================
# 6. Stratified sampling
# ============================================================

print(f"\n{'='*80}")
print("SAMPLING")
print("=" * 80)

rng = np.random.RandomState(RANDOM_SEED)
sampled = []

# A. Sample from full 3-axis bins
if full_traces:
    full_pool = pd.concat(full_traces, ignore_index=True)
    for bin_name, group in full_pool.groupby('physics_bin'):
        bin_samples = []
        for ds_name, ds_group in group.groupby('dataset'):
            n = min(MAX_PER_DATASET_PER_BIN, len(ds_group))
            bin_samples.append(ds_group.sample(n=n, random_state=rng))
        
        bin_df = pd.concat(bin_samples)
        if len(bin_df) > TARGET_PER_BIN:
            bin_df = bin_df.sample(n=TARGET_PER_BIN, random_state=rng)
        
        sampled.append(bin_df)
    print(f"Full 3-axis bins: {sum(len(s) for s in sampled)} traces sampled")

# B. Sample from NEIC (distance × magnitude bins)
neic_count = 0
if neic_traces is not None:
    for bin_name, group in neic_traces.groupby('physics_bin'):
        n = min(NEIC_TARGET_PER_BIN, len(group))
        sampled.append(group.sample(n=n, random_state=rng))
        neic_count += n
    print(f"NEIC (dist × mag): {neic_count} traces across {neic_traces['physics_bin'].nunique()} bins")

# C. Sample from Iquique (distance × depth bins)
iq_count = 0
if iq_traces is not None:
    for bin_name, group in iq_traces.groupby('physics_bin'):
        n = min(IQUIQUE_TARGET_PER_BIN, len(group))
        sampled.append(group.sample(n=n, random_state=rng))
        iq_count += n
    print(f"Iquique (dist × depth): {iq_count} traces across {iq_traces['physics_bin'].nunique()} bins")

# D. Flat sample from OBST2024
if obst_traces is not None:
    n_obs = min(OBST2024_FLAT_SAMPLE, len(obst_traces))
    sampled.append(obst_traces.sample(n=n_obs, random_state=rng))
    print(f"OBST2024 flat: {n_obs} traces")

benchmark = pd.concat(sampled, ignore_index=True)


# ============================================================
# 7. Summary
# ============================================================

print(f"\n{'='*80}")
print(f"BENCHMARK MANIFEST SUMMARY")
print(f"{'='*80}")
print(f"Total traces: {len(benchmark):,}")

print(f"\nBy dataset:")
ds_summary = benchmark.groupby('dataset').agg(
    count=('dataset', 'size'),
    has_s=('has_s_pick', 'sum'),
).sort_values('count', ascending=False)
for idx, row in ds_summary.iterrows():
    print(f"  {idx:20s}  {row['count']:>5}  ({int(row['has_s'])} with S pick)")

print(f"\nBy tectonic type:")
for tt, count in benchmark['tectonic_type'].value_counts().items():
    print(f"  {tt:30s}  {count:>5}")

print(f"\nBy physics bin (top 25):")
for pb, count in benchmark['physics_bin'].value_counts().head(25).items():
    print(f"  {pb:70s}  {count:>5}")

print(f"\nBy distance bin:")
for db in DIST_LABELS:
    n = (benchmark['dist_bin'] == db).sum()
    print(f"  {db:35s}  {n:>5}")
print(f"  {'missing':35s}  {benchmark['dist_bin'].isna().sum():>5}")

print(f"\nBy depth bin:")
for db in DEPTH_LABELS:
    n = (benchmark['depth_bin'] == db).sum()
    print(f"  {db:35s}  {n:>5}")
print(f"  {'missing (NEIC + OBS)':35s}  {benchmark['depth_bin'].isna().sum():>5}")

print(f"\nBy magnitude bin:")
for mb in MAG_LABELS:
    n = (benchmark['mag_bin'] == mb).sum()
    print(f"  {mb:35s}  {n:>5}")
print(f"  {'missing (Iquique + OBS)':35s}  {benchmark['mag_bin'].isna().sum():>5}")

# ============================================================
# 8. Save
# ============================================================

output_cols = [
    'dataset', 'original_index', 'tectonic_type', 'region', 'trained_models',
    'magnitude', 'depth_km', 'distance_km',
    'p_arrival_sample', 's_arrival_sample', 'has_s_pick',
    'dist_bin', 'depth_bin', 'mag_bin', 'physics_bin',
]
benchmark[output_cols].to_csv('benchmark_manifest.csv', index=False)
print(f"\nSaved to benchmark_manifest.csv")
print(f"\nUsage: when evaluating PhaseNet/stead, filter with:")
print(f"  df[~df['trained_models'].str.contains('stead')]")

PROCESSING FULL DATASETS (distance + depth + magnitude)
Reading stead... 1,030,231 valid, 1,023,286 fully binned
Reading scedc... 3,783,102 valid, 3,727,167 fully binned
Reading instancecounts... 713,883 valid, 713,883 fully binned
Reading pnw... 183,909 valid, 159,090 fully binned
Reading txed... 312,231 valid, 311,872 fully binned
Reading geofon... 274,474 valid, 268,004 fully binned
Reading lendb... 629,095 valid, 628,188 fully binned

PROCESSING NEIC (distance × magnitude, depth unknown)
Reading neic... 1,025,000 valid, 1,025,000 binned (dist × mag)

PROCESSING IQUIQUE (distance × depth, magnitude unknown)
Reading iquique... 11,288 valid, 11,288 binned (dist × depth)

PROCESSING OBST2024 (flat sample, no physics metadata)
35,394 valid traces


/tmp/ipykernel_2103959/1559197603.py:429: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pool = pd.concat(all_pools, ignore_index=True)
/tmp/ipykernel_2103959/1559197603.py:429: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pool = pd.concat(all_pools, ignore_index=True)
/tmp/ipykernel_2103959/1559197603.py:429: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining


TOTAL POOL: 7,903,172 traces

PHYSICS BIN POPULATIONS:
  local (<150km) | upper-crustal (0-15km) | micro (M<3)                   n=5,258,096  from 7 ds: geofon, instancecounts, lendb, pnw, scedc, stead, txed
  local (<150km) | lower-crust (15-70km) | micro (M<3)                    n= 798,483  from 6 ds: instancecounts, lendb, pnw, scedc, stead, txed
  teleseismic (>1500km) | depth_unknown | moderate (M3-6)                 n= 374,068  from 1 ds: neic
  local (<150km) | upper-crustal (0-15km) | moderate (M3-6)               n= 269,530  from 7 ds: geofon, instancecounts, lendb, pnw, scedc, stead, txed
  regional (150-1500km) | depth_unknown | moderate (M3-6)                 n= 226,663  from 1 ds: neic
  local (<150km) | depth_unknown | micro (M<3)                            n= 179,848  from 1 ds: neic
  regional (150-1500km) | depth_unknown | micro (M<3)                     n= 121,511  from 1 ds: neic
  local (<150km) | depth_unknown | moderate (M3-6)                        n= 104,006  f

/tmp/ipykernel_2103959/1559197603.py:499: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  benchmark = pd.concat(sampled, ignore_index=True)
/tmp/ipykernel_2103959/1559197603.py:499: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  benchmark = pd.concat(sampled, ignore_index=True)
/tmp/ipykernel_2103959/1559197603.py:499: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when deter